# Automatic tuning of XGBoost parameters using XGBTune
Thanks to the work of [Romain Picard](https://github.com/MainRo) there is now a package, called [XGBTune](https://github.com/MainRo/xgbtune), to automatically tune the parametrs of [XGBoost](https://xgboost.readthedocs.io/en/latest/parameter.html).
From the GitHub page:

## Tuning steps

The tuning is done in the following steps:

*    compute best round
*    tune max_depth and min_child_weight
*    tune gamma
*    re-compute best round
*    tune subsample and colsample_bytree
*    fine tune subsample and colsample_bytree
*    tune alpha and lambda
*    tune seed

This steps can be repeated several times. By default, two passes are done.

Here we shall use the [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) data as an example.

### Install `XGBTune`

In [ ]:
!pip install xgbtune

### set up the House Prices data

In [ ]:
import pandas  as pd
import xgboost as xgb

#===========================================================================
# read in the data
#===========================================================================
train_data = pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv')
test_data  = pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv')

#===========================================================================
# select some features
#===========================================================================
features = ['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 
        'YearBuilt', 'YearRemodAdd', 'BsmtFinSF1', 'BsmtFinSF2', 
        'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 
        'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 
        'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 
        'TotRmsAbvGrd', 'Fireplaces', 'GarageCars', 'GarageArea', 
        'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 
        'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']

#===========================================================================
#===========================================================================
X_train       = train_data[features]
y_train       = train_data["SalePrice"]
X_test        = test_data[features]

### run `xgbtune`
Here we use the root of the mean squared logarithmic error regression loss (`rmsle`) as per the competition requirements

In [ ]:
from xgbtune import tune_xgb_model
params = {'eval_metric': 'rmsle'}
params, round_count = tune_xgb_model(params, X_train, y_train)

### now fit using the parameters, predict, and write out the `submission.csv` file

In [ ]:
#===========================================================================
# now use the parameters from XGBTune
#===========================================================================
regressor=xgb.XGBRegressor(**params)

regressor.fit(X_train, y_train)

#===========================================================================
# use the fit to predict the prices for the test data
#===========================================================================
predictions = regressor.predict(X_test)

#===========================================================================
# write out CSV submission file
#===========================================================================
output = pd.DataFrame({"Id":test_data.Id, "SalePrice":predictions})
output.to_csv('submission.csv', index=False)